# BMY drum transcriber: training notebook

Trains BMY's own drum transcriber (9 drums: kick, snare, 3 toms, closed and open hi-hat, crash, ride) from **licence-clean** material only:

* **E-GMD** drum recordings (Google, CC BY 4.0): 43 kits, read straight out of Google's archive, a file at a time (no 90 GB download).
* Many performances re-played through **free multi-sampled acoustic kits** (CC0 / CC BY / public domain, from github.com/sfzinstruments: Karoryfer, DrumGizmo, Wilkinson Audio, SM Drums), with a random mix balance each time, so the model hears real miked drums, not just one electronic module. One kit (Naked Drums) is kept out of training, so the scores include a kit the model has never heard.
* Some performances re-played through **BMY's GeneralUser GS kits**.
* **Backing instruments generated here** (bass, keys, guitars, pads, leads), mixed under the drums, then split again with **Demucs** (the separator Groove Lab uses), so the model learns real separation artefacts.

**How to run:** `Runtime › Change runtime type › T4 GPU` (or better), then `Runtime › Run all`. Everything is saved in your Google Drive in **BMY_Transcriber**, so if Colab stops (time limit, closed tab), just **Run all** again and it carries on where it stopped.

**When it finishes**, the last cell shows the scores and saves `bmy_drums.onnx` and `bmy_drums.json` in that Drive folder. Send both to Claude, and they become a third Transcriber option in Groove Lab, next to ADTOF and Magenta.

## 1. Settings
`'kits'` is a 2-hour check of the free acoustic kits (same size as the first trial, so the scores compare directly). `'full'` is the real run (two or three sessions). `'trial'` was the first run (TD-17 and GeneralUser only).

In [ ]:
PRESET = 'kits'           # 'kits' (2 h check), 'full' (the real run) or 'trial' (the first run)

SETTINGS = {
    'trial': dict(train_hours=10, val_hours=1, test_hours=1, steps=15000, p_sfz=0.0),
    'kits':  dict(train_hours=10, val_hours=1, test_hours=1, steps=15000, p_sfz=0.45),
    'full':  dict(train_hours=40, val_hours=2, test_hours=2, steps=40000, p_sfz=0.45),
}[PRESET]
SESSION_HOURS = 3.5       # each step stops after this long and saves; Run all again to continue
WORK = '/content/drive/MyDrive/BMY_Transcriber/' + PRESET
print(PRESET, SETTINGS, '->', WORK)

## 2. Set up (Google Drive, tools, BMY code, SoundFont, E-GMD index)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys, urllib.request
os.makedirs(WORK, exist_ok=True)
subprocess.run('apt-get -qq install -y fluidsynth > /dev/null', shell=True, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'demucs', 'remotezip', 'pretty_midi', 'soundfile', 'onnx', 'onnxruntime'], check=True)
if not os.path.exists('/content/bmy-groove-lab'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/taddmlh/bmy-groove-lab.git', '/content/bmy-groove-lab'], check=True)
else:   # always take the latest code (shallow copy: fetch + reset, not pull)
    subprocess.run('git -C /content/bmy-groove-lab fetch -q --depth 1 origin main && git -C /content/bmy-groove-lab reset -q --hard FETCH_HEAD', shell=True, check=True)
sys.path.insert(0, '/content/bmy-groove-lab/training')
import importlib, sfz_kits, bmy_transcriber as bt; importlib.reload(sfz_kits); importlib.reload(bt)
SF2 = '/content/drive/MyDrive/BMY_Transcriber/GeneralUser-GS.sf2'
if not os.path.exists(SF2):
    urllib.request.urlretrieve(bt.GS_SF2_URL, SF2)
CSV = '/content/drive/MyDrive/BMY_Transcriber/e-gmd-v1.0.0.csv'
if not os.path.exists(CSV):
    urllib.request.urlretrieve(bt.EGMD_CSV, CSV)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none (switch the runtime to a GPU!)')
egmd = bt.EGMD(CSV)
print('E-GMD clips:', len(egmd.rows), '| SoundFont MB:', os.path.getsize(SF2) // 2**20)

## 2b. Free acoustic kits
Downloads the free kits to this session's disk (about 5–10 minutes the first time in each session; nothing goes to your Drive). Skipped in the `'trial'` preset.

In [ ]:
import time
if SETTINGS['p_sfz'] > 0:
    prev = '/content/drive/MyDrive/BMY_Transcriber/kit_previews'
    os.makedirs(prev, exist_ok=True)
    cfg0 = bt.PrepConfig(work=WORK)
    for name, k in sfz_kits.KITS.items():
        t = time.time()
        sfz_kits.fetch_kit(name, '/content/kits')
        n, of, pk = bt.kit_preview(name, cfg0, f'{prev}/{name}.wav')
        print(f"{name:10s} {k['licence']:14s} ready in {time.time() - t:4.0f} s | test groove: {n}/{of} hits, peak {pk:5.1f} dB")
    print(subprocess.run(['du', '-sh', '/content/kits'], capture_output=True, text=True).stdout)
    print('Listen: Drive > BMY_Transcriber > kit_previews')


## 3. Quick check (a few seconds of each step on one clip)

In [ ]:
import numpy as np, time
r = egmd.select('train', 0.01, seed=99)[0]
t = time.time()
hits = bt.midi_hits(egmd.read(r['midi_filename'])); y = bt.read_wav(egmd.read(r['audio_filename']))
print(f"clip {r['id']} ({r['kit_name']}, {float(r['duration']):.1f} s): {len(hits)} hits, read in {time.time() - t:.1f} s")
t = time.time(); back = bt.render_backing(10, float(r['bpm']), 1, SF2); print(f'backing rendered in {time.time() - t:.1f} s, rms {np.sqrt(np.mean(back**2)):.3f}')
t = time.time(); stem = bt.DEMUCS(bt.normalise(y[:bt.SR * 10] + back[:bt.SR * 10] * 0.5), DEVICE); print(f'Demucs on 10 s: {time.time() - t:.1f} s')
x = bt.features(bt.normalise(stem)); print('features', x.shape, 'targets', bt.targets(hits, len(x)).shape)
if SETTINGS['p_sfz'] > 0:
    import random
    cfg0 = bt.PrepConfig(work=WORK)
    t = time.time(); kd, played = bt.drums_through_sfz(hits, float(r['duration']) + 1, random.Random(1), cfg0)
    print(f'free-kit render: {len(played)} of {len(hits)} hits in {time.time() - t:.1f} s (includes loading the kit)')


## 4. Prepare the data
The longest step (trial: about 1 hour, full: about 4–5 hours, over two sessions). Safe to stop and re-run: finished clips are kept.

In [ ]:
cfg = bt.PrepConfig(work=WORK, sf2=SF2, train_hours=SETTINGS['train_hours'], val_hours=SETTINGS['val_hours'], test_hours=SETTINGS['test_hours'],
                     p_sfz_kit=SETTINGS['p_sfz'])
budget = SESSION_HOURS * 3600
t0 = time.time()
for split in ('validation', 'test', 'train'):
    left = budget - (time.time() - t0)
    if left < 300:
        print('Session time used up: Run all again to continue.'); break
    bt.prepare(split, cfg, egmd, device=DEVICE, limit_s=left)

## 5. Train
Trial: about 30–45 minutes on a T4. Saves a checkpoint every 2000 steps; re-running continues.

In [ ]:
tcfg = bt.TrainConfig(work=WORK, steps=SETTINGS['steps'])
bt.train(tcfg, device=DEVICE, time_limit_s=SESSION_HOURS * 3600)

## 6. Scores and export
Tunes a threshold per drum on the validation set, scores the held-out test set, and writes `bmy_drums.onnx` + `bmy_drums.json` to the Drive folder.

In [ ]:
import json
meta = bt.export(WORK, device=DEVICE)
print()
print(f"{'drum':10s} {'F1':>6s} {'precision':>10s} {'recall':>8s} {'hits':>6s}")
for k, v in meta.get('test', {}).items():
    print(f"{k:10s} {v['f1']:6.3f} {v['precision']:10.3f} {v['recall']:8.3f} {v.get('n', ''):>6}")
print('\nFiles in', WORK, ':', [f for f in os.listdir(WORK) if f.startswith('bmy_drums')])